In [ ]:
from langchain_community.document_loaders import CSVLoader
from langchain.docstore.document import Document 
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.text_splitter import RecursiveCharacterTextSplitter
import csv
from langchain.embeddings import HuggingFaceEmbeddings
import chromadb
from chromadb.config import Settings

model="llama3.1"
llm = ChatOllama(model=model, temperature=0)


columns_to_metadata=["Match Name", "Match Date","Team1 Name", "Team1 Runs Scored","Team1 Wickets Fell","Team2 Name","Team2 Runs Scored","Team2 Wickets Fell","Match Winner","Match Result Text"]
columns_to_embed = ["Match Name"]

# columns_to_embed=["Match Name","Series Name", "Match Date","Team1 Name", "Team1 Runs Scored","Team1 Wickets Fell","Team2 Name","Team2 Runs Scored","Team2 Wickets Fell","Match Venue (Stadium)","Match Venue (City)","Match Venue (Country)","Umpire 1","Umpire 2","Match Referee","Toss Winner","Toss Winner Choice","Match Winner","Match Result Text"]
# columns_to_metadata = ["Match Name","Series Name", "Match Date","Team1 Name", "Team1 Runs Scored","Team1 Wickets Fell","Team2 Name","Team2 Runs Scored","Team2 Wickets Fell","Match Venue (Stadium)","Match Venue (City)","Match Venue (Country)","Umpire 1","Umpire 2","Match Referee","Toss Winner","Toss Winner Choice","Match Winner","Match Result Text"]

docs = []
print('before',docs)
with open('t20i_Matches_Data.csv', newline="", encoding='utf-8-sig') as csvfile:
    csv_reader = csv.DictReader(csvfile)
    for i, row in enumerate(csv_reader):
        to_metadata = {col: row[col] for col in columns_to_metadata if col in row}
        values_to_embed = {k: row[k] for k in columns_to_embed if k in row}
        to_embed = "\n".join(f"{k.strip()}: {v.strip()}" for k, v in values_to_embed.items())
        newDoc = Document(page_content=to_embed, metadata=to_metadata)
        docs.append(newDoc)
        


ollm_embed = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
chroma_client = chromadb.HttpClient(host="localhost", port = 8000, settings=Settings(allow_reset=True, anonymized_telemetry=False))
chroma_client.delete_collection(name="cricket_match_collection")
vectorstore = Chroma.from_documents(
    documents=docs,
    client=chroma_client,
    collection_name="cricket_match_collection",
    embedding=ollm_embed,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

message = """
You are an AI language model tasked with answering questions based on a specific context provided below. Make sure your answer is directly derived from the context and avoid adding any information not found within it.

Question: {question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm


In [ ]:
# query="who lost in the match India Vs Australia 5Th T20I"
# query="how many wickets did the winning team loose in the match India Vs Australia 5Th T20I"
# query="what is the score of the winning team in the match India Vs Australia 5Th T20I" 
# query="what is the score of the India in the match India Vs Australia 5Th T20I"
# query="how many matches did India play against Australia in the year 2023" # fail
query="in which match did India score run exactly equal to 160"
# Retrieve context
# context_docs = retriever.get_relevant_documents(query)
# print(context_docs)
# Join documents into a single string
# context = "".join([doc.page_content for doc in context_docs])

# Debug: Print context to ensure it's correct
# print("Retrieved Context:")
# print(context)

response = rag_chain.invoke(query)

print(response.content)